# Phase 3 demo: ARM → PicoRV32 → matmul unit

Put these files in the same directory as this notebook on the board:

| File | From |
|---|---|
| `picorv32.bit`, `picorv32.hwh` | `RISCV-on-PYNQ-Z1/build/output/` (`scripts/build_bitstream.sh`) |
| `matmul_fw.bin` | `firmware/matmul/` (`make`) |
| `pynq_matmul.py` | `driver/` |

Flow per call: the ARM allocates A/B/C in DDR and passes their physical addresses to the
PicoRV32 through the BRAM mailbox; the PicoRV32 firmware programs the matmul CSRs for every
8×8×8 job, polls for completion and reports back; the ARM reads C and compares with NumPy.

In [ ]:
import numpy as np
from pynq_matmul import MatmulOverlay, golden

mm = MatmulOverlay("picorv32.bit", "matmul_fw.bin")      # load overlay + firmware

rng = np.random.default_rng(0)
A = rng.integers(-128, 128, (64, 8, 8), dtype=np.int8)   # 64 independent 8x8 tiles
B = rng.integers(-128, 128, (64, 8, 8), dtype=np.int8)

C, stats = mm.matmul(A, B)                                # ARM -> PicoRV32 -> matmul unit
assert np.array_equal(C, golden(A, B)), "hardware result differs from NumPy"
print("PASS:", stats)

## Regression and latency baseline

Random batches through the hardware path vs NumPy (plan §6.3). The per-job cycle counts are
the CSR-path baseline for the Phase 4/6 comparison with the custom-instruction path.

In [ ]:
from pynq_matmul import regression

passed, total, stats = regression(mm, batches=10, batch_size=256, seed=1)
print(f"{passed}/{total} matrices match NumPy ({100 * passed / total:.2f} %)")
print(f"per 8x8x8 job: {stats['riscv_cycles_per_job']:.0f} RISC-V cycles "
      f"({stats['us_per_job']:.2f} us @ 50 MHz), of which accelerator {stats['accel_cycles_per_job']:.0f}")

In [ ]:
# corner cases: int8 extremes
for name, a, b in [("-128 x -128", -128, -128), ("-128 x 127", -128, 127), ("127 x 127", 127, 127)]:
    A1 = np.full((1, 8, 8), a, np.int8); B1 = np.full((1, 8, 8), b, np.int8)
    C1, _ = mm.matmul(A1, B1)
    assert np.array_equal(C1, golden(A1, B1)), name
    print(f"{name:12} -> C[0,0] = {C1[0, 0, 0]}")